# Day 29 Tutorial：图、邻接矩阵与消息求和

## Goal

使用 NumPy 把同一张四节点无向图表示为边列表和邻接矩阵，计算节点度数与邻居特征之和。本教程只验证图表示，不训练模型；输出不是个人练习或下游任务研究结果。

## Setup

输入是人为构造的四个节点、四条无向边和两个教学特征。固定随机种子只是建立可复现习惯；本教程没有随机计算。

In [1]:
import numpy as np

SEED = 29
np.random.seed(SEED)

node_names = ['A', 'B', 'C', 'D']
undirected_edges = [(0, 1), (1, 2), (2, 3), (3, 0)]
node_features = np.array(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
        [0.5, 0.5],
    ],
    dtype=float,
)

print('节点:', node_names)
print('无向边:', undirected_edges)
print('node_features shape:', node_features.shape)
print('node_features dtype:', node_features.dtype)

节点: ['A', 'B', 'C', 'D']
无向边: [(0, 1), (1, 2), (2, 3), (3, 0)]
node_features shape: (4, 2)
node_features dtype: float64


## Steps

先创建 `(4, 4)` 全零矩阵。每条无向边同时写入 `[source, target]` 和 `[target, source]`，所以矩阵应当对称。

In [2]:
num_nodes = len(node_names)
adjacency = np.zeros((num_nodes, num_nodes), dtype=int)

for source, target in undirected_edges:
    adjacency[source, target] = 1
    adjacency[target, source] = 1

print('adjacency shape:', adjacency.shape)
print(adjacency)

adjacency shape: (4, 4)
[[0 1 0 1]
 [1 0 1 0]
 [0 1 0 1]
 [1 0 1 0]]


对邻接矩阵逐行求和得到每个节点的度数。矩阵乘法 `(4, 4) @ (4, 2) → (4, 2)` 会为每个节点加总所有邻居的两个特征。

In [3]:
degree = adjacency.sum(axis=1)
neighbor_message_sum = adjacency @ node_features

neighbor_table = {}
for node_index, node_name in enumerate(node_names):
    neighbor_indices = np.flatnonzero(adjacency[node_index])
    neighbor_table[node_name] = [node_names[index] for index in neighbor_indices]

print('degree shape:', degree.shape)
print('degree:', degree.tolist())
print('neighbors:', neighbor_table)
print('neighbor_message_sum shape:', neighbor_message_sum.shape)
print(neighbor_message_sum)

degree shape: (4,)
degree: [2, 2, 2, 2]
neighbors: {'A': ['B', 'D'], 'B': ['A', 'C'], 'C': ['B', 'D'], 'D': ['A', 'C']}
neighbor_message_sum shape: (4, 2)
[[0.5 1.5]
 [2.  1. ]
 [0.5 1.5]
 [2.  1. ]]


## Checks

检查 shape、整数 dtype、无向对称性、无自环和 A 节点的纸笔消息。A 的邻居是 B 与 D，因此它应收到 `[0.0, 1.0] + [0.5, 0.5] = [0.5, 1.5]`。

In [4]:
assert adjacency.shape == (4, 4)
assert np.issubdtype(adjacency.dtype, np.integer)
assert np.array_equal(adjacency, adjacency.T)
assert np.all(np.diag(adjacency) == 0)
assert np.array_equal(degree, np.array([2, 2, 2, 2]))
assert neighbor_message_sum.shape == (4, 2)

manual_message_for_a = node_features[1] + node_features[3]
assert np.allclose(manual_message_for_a, [0.5, 1.5])
assert np.allclose(neighbor_message_sum[0], manual_message_for_a)

print('全部检查通过。')
print('A 的手算消息:', manual_message_for_a.tolist())
print('边界：这里只验证玩具图表示，没有训练 GNN。')

全部检查通过。
A 的手算消息: [0.5, 1.5]
边界：这里只验证玩具图表示，没有训练 GNN。


## Next Steps

关闭教程，独立完成 `03_exercises.md`。通过后进入 Day 30，把同一张四节点图装入 PyG `Data`；不要把课程预存输出当成自己的学习证据。